# Big Data Lab 1 - Exercise 3
Recommendation: ALS Recommender System


In [5]:
# Exercise 0: Prepare Movie Data
import findspark
import os
import sys
os.environ["JAVA_HOME"] = "C:\\Program Files\\Microsoft\\jdk-11.0.16.101-hotspot"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, explode
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Initialize Spark session
spark = SparkSession.builder \
    .appName("BigData_Lab1") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1") \
    .getOrCreate()

kafka_bootstrap_servers = "localhost:30090,localhost:30091,localhost:30092"

# Define schemas
movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", IntegerType(), True)
])

tags_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", IntegerType(), True)
])

# Read movies from Kafka
df_movies_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", "Lab1_movies") \
    .option("startingOffsets", "earliest") \
    .load()
df_movies = df_movies_kafka.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), movies_schema).alias("data")) \
    .select("data.*")

# Read ratings from Kafka
df_ratings_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", "Lab1_ratings") \
    .option("startingOffsets", "earliest") \
    .load()
df_ratings = df_ratings_kafka.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), ratings_schema).alias("data")) \
    .select("data.*")

# Read tags from Kafka
df_tags_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", "Lab1_tags") \
    .option("startingOffsets", "earliest") \
    .load()
df_tags = df_tags_kafka.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), tags_schema).alias("data")) \
    .select("data.*")

# Show schemas and sample data
df_movies.printSchema()
df_ratings.printSchema()
df_tags.printSchema()



root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: integer (nullable = true)



## Train ALS Model


In [6]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

train_ratings, test_ratings = df_ratings.randomSplit([0.8, 0.2], seed=42)

# Build ALS model
als = ALS(maxIter=10, regParam=0.1, userCol="userId", itemCol="movieId", ratingCol="rating", coldStartStrategy="drop")
model_als = als.fit(train_ratings)
predictions = model_als.transform(test_ratings)



## Evaluate Model (RMSE)


In [7]:
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print(f"RMSE on test data: {rmse}")



RMSE on test data: 0.880792066913646


## Evaluate Model (Precision@10)


In [8]:
from pyspark.sql.functions import expr, avg, collect_list, explode
from pyspark.sql.types import FloatType
from pyspark.sql.functions import udf

# Get relevant items from ground truth (rating >= 4)
relevant_items = test_ratings.filter(col("rating") >= 4).groupBy("userId").agg(collect_list("movieId").alias("rel_movies"))

# Recommend top 10 for all users
userRecs = model_als.recommendForAllUsers(10)

# Join relevant items and recommendations
eval_df = relevant_items.join(userRecs, on="userId")

def calc_precision(rel_movies, recs):
    if not rel_movies or not recs: return 0.0
    rec_movie_ids = [r.movieId for r in recs]
    hits = len(set(rel_movies).intersection(set(rec_movie_ids)))
    return hits / 10.0

precision_udf = udf(calc_precision, FloatType())
eval_df = eval_df.withColumn("precision_at_10", precision_udf("rel_movies", "recommendations"))

mean_precision = eval_df.select(avg("precision_at_10")).collect()[0][0]
print(f"Precision@10 (averaged across users with >= 1 relevant item): {mean_precision:.4f}")



Py4JJavaError: An error occurred while calling o479.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 321.0 failed 1 times, most recent failure: Lost task 0.0 in stage 321.0 (TID 741) (DESKTOP-2954F58 executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:99)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:75)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.

In [9]:
from pyspark.sql.functions import col, expr, size, array_intersect, avg

# 1. Join relevant items and recommendations (Same as your code)
eval_df = relevant_items.join(userRecs, on="userId")

# 2. Extract movie IDs from the recommendations struct array using 'transform'
eval_df = eval_df.withColumn(
    "rec_movie_ids", 
    expr("transform(recommendations, x -> x.movieId)")
)

# 3. Find the intersection between relevant movies and recommended movies
eval_df = eval_df.withColumn(
    "hits", 
    size(array_intersect(col("rel_movies"), col("rec_movie_ids")))
)

# 4. Calculate Precision@10
eval_df = eval_df.withColumn(
    "precision_at_10", 
    col("hits") / 10.0
)

# 5. Collect the final average
mean_precision = eval_df.select(avg("precision_at_10")).collect()[0][0]
print(f"Precision@10 (averaged across users with >= 1 relevant item): {mean_precision:.4f}")

Precision@10 (averaged across users with >= 1 relevant item): 0.0014


## Top 10 Recommendations for 3 Random Users


In [10]:
# Select 3 random users
random_users = df_ratings.select("userId").distinct().sample(False, 0.1, seed=42).limit(3)

# Generate recommendations
recs_3_users = model_als.recommendForUserSubset(random_users, 10)

# Display results
recs_3_users_list = recs_3_users.collect()
for row in recs_3_users_list:
    print(f"\nRecommendations for User ID: {row.userId}")
    for idx, rec in enumerate(row.recommendations, 1):
        print(f"  {idx}. Movie ID: {rec.movieId} (Predicted Rating: {rec.rating:.2f})")




Recommendations for User ID: 31
  1. Movie ID: 6380 (Predicted Rating: 5.31)
  2. Movie ID: 2035 (Predicted Rating: 5.15)
  3. Movie ID: 93008 (Predicted Rating: 5.14)
  4. Movie ID: 77846 (Predicted Rating: 5.14)
  5. Movie ID: 25906 (Predicted Rating: 5.14)
  6. Movie ID: 122886 (Predicted Rating: 5.09)
  7. Movie ID: 1939 (Predicted Rating: 5.07)
  8. Movie ID: 1366 (Predicted Rating: 5.04)
  9. Movie ID: 88125 (Predicted Rating: 5.02)
  10. Movie ID: 93510 (Predicted Rating: 5.01)

Recommendations for User ID: 481
  1. Movie ID: 51931 (Predicted Rating: 5.03)
  2. Movie ID: 1939 (Predicted Rating: 4.76)
  3. Movie ID: 3451 (Predicted Rating: 4.76)
  4. Movie ID: 3022 (Predicted Rating: 4.70)
  5. Movie ID: 47423 (Predicted Rating: 4.70)
  6. Movie ID: 2135 (Predicted Rating: 4.69)
  7. Movie ID: 4429 (Predicted Rating: 4.64)
  8. Movie ID: 2398 (Predicted Rating: 4.59)
  9. Movie ID: 940 (Predicted Rating: 4.53)
  10. Movie ID: 910 (Predicted Rating: 4.51)

Recommendations for Use